# 💳 Task 1: Credit Scoring Model
**CodeAlpha Machine Learning Internship**

### Objective
Predict an individual's creditworthiness using past financial data.

### Approach
- Logistic Regression
- Decision Tree
- Random Forest

### Metrics
- Accuracy, Precision, Recall, F1-Score, ROC-AUC

In [ ]:
# ─── Install required libraries (run once) ───
# !pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# ─── 1. Import Libraries ───
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

In [ ]:
# ─── 2. Load Dataset ───
# Option A: Use built-in synthetic data (no download needed)
# Option B: Uncomment below to load German Credit Dataset from UCI

# --- Option A: Synthetic Dataset ---
np.random.seed(42)
n = 1000

data = pd.DataFrame({
    'age':              np.random.randint(18, 70, n),
    'income':          np.random.randint(15000, 120000, n),
    'loan_amount':     np.random.randint(1000, 50000, n),
    'loan_duration':   np.random.randint(6, 60, n),
    'num_credits':     np.random.randint(1, 10, n),
    'employment_years':np.random.randint(0, 30, n),
    'debt_ratio':      np.round(np.random.uniform(0.05, 0.9, n), 2),
    'payment_history': np.random.choice([0, 1], n, p=[0.3, 0.7]),  # 1=good
    'credit_score':    np.random.randint(300, 850, n),
})

# Target: 1 = Good Credit, 0 = Bad Credit
data['creditworthy'] = ((data['credit_score'] > 600) &
                         (data['debt_ratio'] < 0.5) &
                         (data['payment_history'] == 1)).astype(int)

print(f'Dataset shape: {data.shape}')
print(f"\nClass distribution:\n{data['creditworthy'].value_counts()}")
data.head()

In [ ]:
# ─── 3. Exploratory Data Analysis ───
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Credit Score Distribution
axes[0].hist(data['credit_score'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Credit Score Distribution')
axes[0].set_xlabel('Credit Score')

# Debt Ratio by Class
for label, grp in data.groupby('creditworthy'):
    axes[1].hist(grp['debt_ratio'], bins=20, alpha=0.6,
                 label='Good' if label==1 else 'Bad')
axes[1].set_title('Debt Ratio by Creditworthiness')
axes[1].legend()

# Class balance
data['creditworthy'].value_counts().plot(kind='bar', ax=axes[2],
    color=['salmon', 'steelblue'], edgecolor='white')
axes[2].set_title('Class Balance')
axes[2].set_xticklabels(['Bad (0)', 'Good (1)'], rotation=0)

plt.tight_layout()
plt.savefig('credit_eda.png', dpi=100, bbox_inches='tight')
plt.show()
print('EDA plots saved!')

In [ ]:
# ─── 4. Feature Engineering & Preprocessing ───
X = data.drop('creditworthy', axis=1)
y = data['creditworthy']

# Feature: loan-to-income ratio
X['loan_to_income'] = X['loan_amount'] / X['income']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
# ─── 5. Train Models ───
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc':  roc_auc_score(y_test, y_prob)
    }
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"Accuracy : {results[name]['accuracy']:.4f}")
    print(f"ROC-AUC  : {results[name]['roc_auc']:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Bad','Good']))

In [ ]:
# ─── 6. ROC Curve Comparison ───
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Credit Scoring')
plt.legend()
plt.tight_layout()
plt.savefig('credit_roc.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 7. Feature Importance (Random Forest) ───
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(8, 5),
                                color='steelblue', edgecolor='white')
plt.title('Feature Importances — Random Forest')
plt.tight_layout()
plt.savefig('credit_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 8. Confusion Matrix — Best Model ───
best_name = max(results, key=lambda k: results[k]['roc_auc'])
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Bad','Good'], yticklabels=['Bad','Good'])
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('credit_confusion.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'\nBest Model: {best_name} (ROC-AUC = {results[best_name]["roc_auc"]:.4f})')

## ✅ Summary
| Model | Accuracy | ROC-AUC |
|---|---|---|
| Logistic Regression | 0.9350| 0.9761 |
| Decision Tree | 1.0000 | 1.OOOO |
| Random Forest | 1.0000| 1.0000|

**Random Forest** typically performs best due to ensemble learning.

**GitHub Repo:** `CodeAlpha_CreditScoringModel`